# Adaptation QLoRA du tuteur Brainmaxing (bloc C)

> **Pour le rendu :** menu **Exécution > Tout exécuter**, attendre la fin (environ une à deux heures sur A100), puis **Fichier > Enregistrer** pour conserver les sorties dans le notebook. **§7.5 à rédiger à la main, puis enregistrer** : les commentaires d'analyse de cas ne sont pas écrits à l'avance.

## Résumé

- **Objectif** : apprendre à Qwen2.5-3B-Instruct à jouer le tuteur. Il lit la réponse d'un élève, donne un verdict (`PARFAIT`, `PARTIEL`, `INCORRECT`) ou refuse (`HORS_PERIMETRE`), puis écrit un feedback qui ne donne pas la réponse.
- **Méthode** : QLoRA. Le modèle est compressé en 4 bits et gelé ; on entraîne seulement de petites matrices ajoutées à côté (les adaptateurs LoRA).
- **Suivi** : les pertes (loss) d'entraînement et de validation sont relevées pendant l'entraînement ; l'adaptateur qui a la meilleure loss de validation est conservé.
- **Évaluation** : on compare le modèle de base et le modèle adapté sur le verdict, puis on lit des cas concrets, réussites et erreurs.
- **Durée et prérequis** : environ une à deux heures sur un GPU A100 (une L4 ou une T4 fonctionnent, en bien plus lent ; la durée réelle de l'entraînement est affichée au §5). Il faut un compte Google avec Drive : une session Colab peut être coupée, et tout ce qui n'est pas sur Drive est perdu.

## Ce que le notebook produit

| Produit | Rôle | Où |
|---|---|---|
| `best/` | meilleur adaptateur LoRA selon la loss de validation | `SAVE_DIR/v1.0/best` |
| `config.json`, `resultat.json` | réglages utilisés, meilleur pas, raison de l'arrêt, durée | `SAVE_DIR/v1.0` |
| `historique_pas.csv`, `historique_val.csv`, `historique_loss.csv` | loss train par pas, loss val par mesure, résumé par epoch | `SAVE_DIR/v1.0` |
| courbe de loss (PNG) et tableau `comparaison_runs.md` | suivi de l'entraînement et comparaison avec les essais précédents | `EVAL_DIR` |
| `eval_v1.0_base.json`, `eval_v1.0_lora.json` | verdict attendu et verdict prédit pour chaque exemple de validation | `EVAL_DIR` |
| archive `.zip` de `best/` | adaptateur prêt à être copié ou envoyé | `EVAL_DIR` |
| `cas_figes_v1.0.json` | liste des cas analysés au §7, figée pour que les commentaires restent valables | `EVAL_DIR` |

## Lien avec les autres blocs

- **Bloc A (fiches concept) et dataset** : les exemples d'entraînement viennent des fiches ; leur contenu est injecté dans le prompt, le modèle ne le mémorise pas.
- **Bloc B (baseline)** : le modèle de base, sans adaptateur, reçoit exactement les mêmes prompts que le modèle adapté ; c'est la référence « avant ».
- **Bloc D (évaluation) et bloc E (Gradio)** : D mesure sur un jeu de test indépendant et note la qualité pédagogique ; E charge le dossier `best/` produit ici.

## 1. Installation et reproductibilité

Cette partie prépare la machine : GPU, Drive, code du dépôt, bibliothèques aux versions figées. Tout ce qui suit s'appuie sur les variables de chemin définies ici.

### 1.1 Vérifier le GPU

Le chargement en 4 bits (bibliothèque `bitsandbytes`) exige un GPU Nvidia. On vérifie qu'il est attribué avant de continuer, pour échouer tout de suite. Si la cellule échoue : **Exécution > Modifier le type d'exécution**, choisir un GPU (idéalement A100), puis relancer.

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), "Aucun GPU : Exécution > Modifier le type d'exécution > GPU"
print("GPU :", torch.cuda.get_device_name(0))

### 1.2 Monter Google Drive et définir les chemins

Une fenêtre Google va s'ouvrir pour demander l'autorisation d'accéder au Drive : il faut l'accepter, sinon Colab ne peut pas y écrire.

Les adaptateurs (`SAVE_DIR`) et les évaluations (`EVAL_DIR`) sont écrits directement sur Drive. `RUN_NAME` nomme cet entraînement : `v1.0` est la version finale. Si la session est coupée en cours d'entraînement, il n'y a pas de reprise : on change `RUN_NAME` en `v1.0b` et on relance tout.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "v1.0"  # à changer en "v1.0b" si la session a été coupée (pas de reprise)
SAVE_DIR = "/content/drive/MyDrive/brainmaxing/adapters"
EVAL_DIR = "/content/drive/MyDrive/brainmaxing/eval"

RUN_DIR = f"{SAVE_DIR}/{RUN_NAME}"
BEST_DIR = f"{RUN_DIR}/best"
BASE_JSON = f"{EVAL_DIR}/eval_{RUN_NAME}_base.json"
LORA_JSON = f"{EVAL_DIR}/eval_{RUN_NAME}_lora.json"
PLOT_PATH = f"{EVAL_DIR}/courbe_loss_{RUN_NAME}.png"
BARS_PATH = f"{EVAL_DIR}/exactitude_par_verdict_{RUN_NAME}.png"
RUNS_TABLE = f"{EVAL_DIR}/comparaison_runs.md"
FROZEN_PATH = f"{EVAL_DIR}/cas_figes_{RUN_NAME}.json"
ZIP_BASE = f"{EVAL_DIR}/adaptateur_{RUN_NAME}_best"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
print("adaptateur final :", BEST_DIR)

### 1.3 Récupérer le code

On clone le dépôt seulement s'il n'existe pas déjà (la cellule peut donc être relancée), puis on se place dedans et on récupère les dernières modifications. Les commandes `python -m src.training...` doivent être lancées depuis la racine du dépôt.

In [ ]:
BRANCH = "bloc-c-entrainement"  # remplacer par "main" après la fusion de la branche
REPO_URL = "https://github.com/EmineOA/NLP-Transformers.git"

%cd /content
![ -d NLP-Transformers ] || git clone --branch {BRANCH} {REPO_URL}
%cd /content/NLP-Transformers
!git pull

### 1.4 Installer les bibliothèques

Les versions sont figées dans `requirements-qlora.txt` pour que le notebook donne les mêmes résultats d'une exécution à l'autre. PyTorch est déjà installé sur Colab : on ne le réinstalle pas.

- Si Colab demande de redémarrer la session après `pip`, le faire, puis relancer à partir du §1.2 (les variables sont perdues au redémarrage).
- Pour ré-exécuter seulement l'analyse après un entraînement terminé, commencer au §5 (après avoir relancé §1.2 et §1.3 si la session a changé). Relancer la cellule d'entraînement sur le même `RUN_NAME` est refusé par sécurité.

In [ ]:
!pip install -q -r requirements-qlora.txt

### 1.5 Noter les versions et le commit

Ces lignes font partie du résultat : elles disent avec quoi le notebook a tourné. Le commit identifie exactement la version du code utilisée. La dernière ligne signale un code ancien (sans l'option `--eval-every`) ; si elle s'affiche, il faut mettre à jour le dépôt avant de continuer.

In [ ]:
import platform
import sys
from importlib.metadata import version

print("python", platform.python_version())
for package in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "matplotlib", "pandas"]:
    print(f"{package:<14}{version(package)}")
print("branche :", BRANCH)
!git rev-parse --short HEAD
# l'option --eval-every n'existe que dans le code récent : sa présence prouve que le bon code est cloné
!python -m src.training.run_training --help | grep -q eval-every || echo "ATTENTION : code ancien, faire git pull sur la bonne branche (BRANCH, §1.3)"

### 1.6 Fixer la graine

La graine rend une partie du hasard reproductible. **Seul l'ordre de mélange des exemples est fixé par la graine** (42, définie dans `src/training/train.py`). L'initialisation de LoRA et le dropout ne sont pas fixés : deux exécutions donnent des résultats proches, pas identiques. La génération du §6 est gloutonne (toujours le token le plus probable), donc sans hasard.

In [ ]:
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
sys.path.insert(0, os.getcwd())  # pour que "import src..." trouve le dépôt cloné
print("graine :", SEED, "| dossier de travail :", os.getcwd())

## 2. Les données

Le dataset est composé d'exemples de dialogue pédagogique (les effectifs exacts sont calculés au §2.1). Chaque exemple est composé de trois messages :

1. `system` : la consigne du tuteur, avec la **fiche concept** et la question posée ;
2. `user` : l'historique résumé de l'élève et sa **réponse actuelle** ;
3. `assistant` : la réponse cible, soit `[VERDICT: PARFAIT|PARTIEL|INCORRECT]`, soit `[HORS_PERIMETRE]`, puis le feedback.

Les fiches viennent du bloc A. Les réponses d'élèves et les feedbacks ont été générés par deux LLM (`claude-opus-5.5` et `sol-6.1`) à partir de ces fiches, avec des contrôles automatiques, mais **pas relus ligne à ligne**. Les réponses d'élèves sont fictives : le jeu ne contient aucune donnée personnelle réelle.

Le découpage entre entraînement (`train`) et validation (`val`) est **groupé par question** : une question de `val` n'apparaît jamais dans `train`. La loss de validation mesure donc la capacité à traiter une question inédite, mais, comme on le voit au §2.2, pas un concept inédit.

### 2.1 Effectifs et répartition des verdicts

On charge les deux groupes avec `load_split` et on compte les verdicts attendus. Vérifier l'équilibre des classes est utile : un verdict trop rare serait mal appris.

In [ ]:
import pandas as pd
from src.training.dataset import IGNORE_INDEX, MAX_LENGTH, load_split, token_lengths, tokenize_example
from src.training.evaluate_verdicts import label_of

VERDICTS = ["PARFAIT", "PARTIEL", "INCORRECT", "HORS_PERIMETRE"]

train = load_split("train")
val = load_split("val")
total = len(train) + len(val)
print(f"train : {len(train)} exemples | val : {len(val)} exemples | total : {total}")

# le verdict attendu est lu dans la 1re ligne du message assistant, avec le même parseur que l'évaluation
counts = pd.DataFrame({
    name: pd.Series([label_of(conversation[-1]["content"]) for conversation in conversations]).value_counts()
    for name, conversations in (("train", train), ("val", val))
}).reindex(VERDICTS).fillna(0).astype(int)
counts["total"] = counts.sum(axis=1)
assert counts["total"].sum() == total, "les verdicts comptés ne couvrent pas tous les exemples"
counts

### 2.2 Ce que contient `val`

Le split est décrit dans `dataset_source.csv` (colonnes `question_id`, `concept_id`, `split`). On calcule ici de quelles questions viennent les exemples de `val`, et si leur concept apparaît aussi dans `train` par d'autres questions. Cette information conditionne la lecture de tous les scores de validation.

In [ ]:
from src.training.dataset import DATA_DIR

source = pd.read_csv(DATA_DIR / "dataset_source.csv", sep=";", encoding="utf-8-sig")
val_part = source[source["split"] == "val"]
train_part = source[source["split"] == "train"]

by_question = val_part.groupby(["question_id", "concept_id"]).size().rename("exemples").reset_index()
by_question["concept aussi dans train"] = by_question["concept_id"].isin(set(train_part["concept_id"]))
display(by_question)
print(f"val = {by_question['question_id'].nunique()} question(s) pour {len(val_part)} exemples")
print("questions de val présentes dans train :", bool(set(val_part["question_id"]) & set(train_part["question_id"])))

Lecture : les exemples de `val` sont des **variantes d'un très petit nombre de questions** (verdicts, essais et styles d'élève différents pour une même question). Ils sont donc très corrélés entre eux : seule la **question** est inédite, pas le concept, qui apparaît dans `train` par d'autres questions. Un score sur `val` mesure la généralisation à une nouvelle question d'un concept connu, pas à un nouveau concept.

### 2.3 Un exemple complet

On affiche un exemple de verdict `PARTIEL`. Le `system` est tronqué (il contient toute la fiche concept), le `user` et l'`assistant` sont complets. L'`assistant` est ce que le modèle doit apprendre à écrire.

In [ ]:
example = next(conversation for conversation in train if label_of(conversation[-1]["content"]) == "PARTIEL")
system_message, user_message, assistant_message = example

print("=== SYSTEM (600 premiers caractères sur", len(system_message["content"]), ") ===")
print(system_message["content"][:600], "[...]")
print("\n=== USER ===")
print(user_message["content"])
print("\n=== ASSISTANT (réponse à apprendre) ===")
print(assistant_message["content"])

**Ce que le modèle apprend** : le **format** de sortie (une première ligne avec le verdict, puis le feedback), le **bon verdict** selon la réponse de l'élève, le **style** du feedback (il fait réfléchir sans donner la réponse) et le **refus** des demandes hors sujet.

**Ce qu'il n'apprend pas** : le contenu des fiches. Il est injecté dans le prompt (`system`) à chaque appel. Le modèle apprend à *appliquer* une fiche, pas à la retenir.

### 2.4 Longueur des conversations en tokens

Un token est un morceau de mot. Le modèle lit au plus `max_length` tokens, et `tokenize_example` refuse de couper une conversation (couper enlèverait la fin, c'est-à-dire la réponse à apprendre). On mesure donc les longueurs réelles, avec le tokenizer du modèle, pour vérifier que la limite choisie est suffisante.

In [ ]:
from transformers import AutoTokenizer
from src.llm.model import DEFAULT_MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL_NAME)

stats = {}
for name, conversations in (("train", train), ("val", val)):
    lengths = token_lengths(conversations, tokenizer)
    stats[name] = {"min": min(lengths), "moyenne": round(sum(lengths) / len(lengths)), "max": max(lengths)}
display(pd.DataFrame(stats).T)

longest = max(values["max"] for values in stats.values())
print(f"plus longue conversation : {longest} tokens | max_length retenu : {MAX_LENGTH}")
print("toutes les conversations tiennent :", longest <= MAX_LENGTH)

`max_length = 4096` est une puissance de 2 située au-dessus de la conversation la plus longue mesurée (la cellule précédente le vérifie) : aucune conversation n'est coupée, et la marge évite de tout refaire si on ajoute des exemples un peu plus longs.

### 2.5 Le masquage : la loss ne porte que sur la réponse

`tokenize_example` transforme une conversation en trois listes : `input_ids` (le texte en tokens), `attention_mask` et `labels` (ce que le modèle doit deviner). Dans `labels`, tout le prompt (consigne et message de l'élève) vaut `-100`, valeur que PyTorch ignore dans le calcul de la loss. On montre le résultat sur l'exemple précédent.

In [ ]:
encoded = tokenize_example(example, tokenizer)
learned_tokens = [label for label in encoded["labels"] if label != IGNORE_INDEX]
number_of_masked = len(encoded["labels"]) - len(learned_tokens)

print(f"tokens au total          : {len(encoded['labels'])}")
print(f"masqués (-100, le prompt) : {number_of_masked}")
print(f"appris (réponse du tuteur) : {len(learned_tokens)}")
print("\ntexte décodé de la partie apprise :")
print(tokenizer.decode(learned_tokens))

La loss ne porte que sur la réponse du tuteur parce que c'est la seule partie que le modèle doit produire. Le prompt est donné, il est très long (la fiche entière) et identique d'un exemple à l'autre : l'apprendre par cœur ne servirait à rien et noierait le signal utile. La balise de fin `<|im_end|>` fait partie de la partie apprise : le modèle apprend aussi à s'arrêter.

### 2.6 Limites des données

- **Labels générés** : les verdicts attendus ont été imposés à deux LLM générateurs, puis vérifiés par des contrôles automatiques ; les lignes n'ont pas été relues une à une par plusieurs humains. Un label douteux apprend une erreur au modèle.
- **Validation étroite et corrélée** : `val` ne contient que quelques questions, chacune en de nombreuses variantes (§2.2). Un bon score sur `val` ne garantit pas la même performance sur d'autres concepts.
- **Données synthétiques** : les réponses d'élèves sont fictives, plus propres et plus régulières que de vraies copies.

## 3. Le modèle et la quantification QLoRA

**Quantification (le Q de QLoRA).** Les poids du modèle de base sont stockés en 4 bits au lieu de 16, ce qui divise la mémoire par environ 4. Ils sont **gelés** : l'entraînement ne les modifie jamais.

**LoRA.** À côté de chaque grande matrice gelée `W`, on ajoute deux petites matrices `A` et `B`, seules entraînées :

```
sortie = W·x + (alpha / r) · B · A · x
```

`r` (le rang) est la taille des petites matrices et `alpha / r` règle la force de la correction. Il y a donc très peu de paramètres à entraîner, et l'adaptateur sauvegardé est léger.

### 3.1 Hyperparamètres choisis

| Réglage | Valeur | Pourquoi |
|---|---|---|
| Modèle de base | `Qwen/Qwen2.5-3B-Instruct` | assez petit pour tenir en 4 bits sur un seul GPU, déjà entraîné à suivre des consignes, commun à tous les blocs |
| Quantification | 4 bits `nf4`, double quantification, calcul en bfloat16 (float16 sur T4) | `nf4` place les 16 valeurs possibles là où se trouvent la plupart des poids ; la double quantification économise encore de la mémoire |
| Rang `r` | 16 | valeur courante : assez de capacité pour apprendre un format et un style, adaptateur léger ; non optimisée (voir §9) |
| `alpha` | 32 | `alpha / r = 2`, convention usuelle |
| Dropout LoRA | 0.05 | 5 % des valeurs sont mises à 0 au hasard pendant l'entraînement pour limiter l'apprentissage par cœur |
| Couches ciblées | `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj` | toutes les couches linéaires des blocs Transformer, comme dans l'article QLoRA ; pas `lm_head` |
| Learning rate | 1e-4 | pas modéré ; l'effet est comparé au §5.4 avec l'essai précédent à 2e-4 (valeur par défaut du code) |
| Taille de lot | 1 exemple × 8 accumulations = 8 | un exemple à la fois (conversations longues, pas de remplissage), gradients additionnés sur 8 exemples pour une correction plus stable |
| Époques | 3 au maximum | quelques passages sur le jeu d'entraînement ; au-delà le modèle risque d'apprendre par cœur. L'arrêt anticipé peut raccourcir |
| Longueur maximale | 4096 tokens | la plus longue conversation tient (§2.4) |
| Patience | 4 | arrêt après 4 mesures de loss val de suite sans record : une seule mesure moins bonne peut être du bruit |
| Mesure val tous les | 25 pas | permet de choisir le meilleur adaptateur « au pas près » et pas seulement à la fin d'une epoch ; chaque mesure coûte un peu de temps |
| Optimiseur | AdamW, graine 42 | optimiseur standard ; la graine fixe l'ordre de mélange des exemples |

### 3.2 Contrôle : les constantes du code

Pour que ce tableau ne puisse pas se contredire avec le code, on affiche les constantes réellement utilisées par l'entraînement.

In [ ]:
from src.training.model import LORA_ALPHA, LORA_DROPOUT, LORA_RANK, LORA_TARGET_MODULES
from src.training.train import ACCUMULATION_STEPS, LEARNING_RATE

print("modèle              :", DEFAULT_MODEL_NAME)
print("rang r / alpha      :", LORA_RANK, "/", LORA_ALPHA)
print("dropout LoRA        :", LORA_DROPOUT)
print("couches ciblées     :", ", ".join(LORA_TARGET_MODULES))
print("accumulation        :", ACCUMULATION_STEPS, "(lot effectif =", 1 * ACCUMULATION_STEPS, ")")
print("max_length          :", MAX_LENGTH)
print("learning rate défaut:", LEARNING_RATE, "(le notebook passe 1e-4 en argument au §4)")

### 3.3 Charger le modèle en 4 bits et ajouter LoRA

On charge le modèle quantifié, on ajoute les adaptateurs, puis on mesure deux choses : la mémoire GPU et la proportion de paramètres entraînables. Cette proportion doit être très petite : c'est tout l'intérêt de LoRA.

Cette cellule sert à **inspecter** le modèle. L'entraînement du §4 tourne dans un processus séparé qui recharge tout lui-même.

In [ ]:
from src.training.model import add_lora, load_model_4bit

model, tokenizer = load_model_4bit()
print(f"mémoire GPU après chargement 4 bits : {torch.cuda.memory_allocated() / 1024**3:.2f} Go")

model = add_lora(model)  # rang, alpha et dropout par défaut = ceux du tableau
trainable, total = model.get_nb_trainable_parameters()  # compte correctement les poids 4 bits
print(f"paramètres entraînables : {trainable:,} sur {total:,} ({100 * trainable / total:.2f} %)")
print(f"mémoire GPU après add_lora : {torch.cuda.memory_allocated() / 1024**3:.2f} Go")

### 3.4 Libérer la mémoire

L'entraînement du §4 est lancé dans un autre processus, qui recharge son propre modèle. Si on gardait celui-ci en mémoire, les deux se partageraient le GPU. On le supprime donc.

In [ ]:
import gc

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"mémoire GPU allouée : {torch.cuda.memory_allocated() / 1024**3:.2f} Go")

## 4. Entraînement

### 4.1 Ce que fait l'entraînement

Le script `src/training/run_training.py` corrige `A` et `B` après chaque groupe de 8 exemples (un **pas** d'optimisation). Toutes les 25 pas, et à la fin de chaque epoch, il mesure la loss sur **tout** `val`.

- **Meilleur adaptateur au pas près** : à chaque record de loss de validation, l'adaptateur est écrit dans `best/`. À la fin, `best/` contient le meilleur état rencontré, pas forcément le dernier.
- **Arrêt anticipé** : après 4 mesures de suite sans record (`--patience 4`), l'entraînement s'arrête. Il peut tomber **au milieu d'une epoch** : cette epoch n'est alors ni sauvegardée (`epoch_N/`) ni écrite dans `historique_loss.csv` (epochs complètes seulement) ; elle est décrite dans `resultat.json` (`epoch_partielle`, `pas_dans_epoch`).
- **Pas d'écrasement** : le script refuse un dossier de sortie déjà utilisé. En cas de relance, on change `RUN_NAME` en `v1.0b` (§1.2) ; il n'y a pas de reprise.

### 4.2 Lancer l'entraînement

La sortie est filtrée : le script écrit une ligne par exemple (des milliers de lignes au total), sans intérêt à lire. Elles sont retirées de l'affichage ; la loss de chaque pas reste dans `historique_pas.csv`. Restent les mesures de validation, les résumés d'epoch complètes et, si l'arrêt anticipé tombe en cours d'epoch, la ligne « epoch N interrompue ». La durée dépend du GPU ; la durée réelle est affichée au §5.

In [ ]:
# En cas de relance (session coupée, dossier déjà utilisé) : changer RUN_NAME en "v1.0b" au §1.2
!python -u -m src.training.run_training --epochs 3 --learning-rate 1e-4 --patience 4 --eval-every 25 --output-dir {RUN_DIR} 2>&1 | grep --line-buffered -v "^epoch [0-9]*, exemple"

### 4.3 Contrôler que l'entraînement est terminé

`resultat.json` n'est écrit qu'à la fin d'un entraînement réussi. Si la cellule suivante échoue, il faut relire la sortie de la cellule précédente.

In [ ]:
assert os.path.exists(f"{RUN_DIR}/resultat.json"), "l'entraînement a échoué : relire la sortie de la cellule précédente"
print("entraînement terminé :", RUN_DIR)

## 5. Suivi de l'entraînement

### 5.1 Réglages et résultat

`config.json` est écrit **avant** l'entraînement (réglages utilisés) et `resultat.json` **après** (meilleur pas, raison de l'arrêt, durée). Les deux sont lus depuis Drive et non recopiés à la main.

In [ ]:
import json

with open(f"{RUN_DIR}/config.json", encoding="utf-8") as file:
    config = json.load(file)
with open(f"{RUN_DIR}/resultat.json", encoding="utf-8") as file:
    result = json.load(file)

config["lora_target_modules"] = ", ".join(config["lora_target_modules"])
display(pd.DataFrame({"valeur": config}).rename_axis("config.json"))
display(pd.DataFrame({"valeur": result}).rename_axis("resultat.json"))

### 5.2 Courbes de loss

On trace la loss d'entraînement (une valeur par pas, avec une moyenne glissante pour lire la tendance) et la loss de validation (une valeur par mesure). Le meilleur point de validation est entouré. Le PNG est écrit sur Drive puis affiché ici.

In [ ]:
from IPython.display import Image
from src.training.plot_losses import plot_training_run

plot_training_run(RUN_DIR, PLOT_PATH, f"Loss d'entraînement et de validation par pas d'optimisation ({RUN_NAME})")
Image(PLOT_PATH)

### 5.3 Comment lire la courbe

- **Train (bleu) et val (orange)** : la courbe bleue (erreur sur les exemples appris, bruitée car chaque point moyenne 8 exemples) doit descendre ; la courbe orange (erreur sur des exemples jamais vus) sert à choisir l'adaptateur.
- **Val remonte alors que train baisse** : surapprentissage, le modèle récite ses exemples. De petites oscillations de val ne suffisent pas à conclure, d'où la patience de plusieurs mesures.
- **Arrêt** : on garde l'adaptateur du pas où la loss val est minimale. L'arrêt peut tomber en cours d'epoch : `historique_loss.csv` et le §5.4 ne comptent alors que les epochs terminées, alors que la courbe montre tous les pas.

La cellule suivante vérifie comment l'arrêt s'est passé, avec les données réelles.

In [ ]:
val_history = pd.read_csv(f"{RUN_DIR}/historique_val.csv")  # colonnes : pas, epoch, loss_val
best_row = val_history.loc[val_history["loss_val"].idxmin()]
after_best = val_history[val_history["pas"] > best_row["pas"]]

print(f"mesures de validation : {len(val_history)}")
print(f"meilleure mesure      : pas {int(best_row['pas'])} (epoch {int(best_row['epoch'])}), loss val {best_row['loss_val']:.4f}")
print(f"mesures après le record : {len(after_best)} | pas total : {result['pas_total']}")
print(f"durée de l'entraînement : {result['duree_totale_secondes'] / 60:.0f} min ({config['gpu']})")
print(f"epoch interrompue ? {result['epoch_partielle']} ({result['pas_dans_epoch']} pas faits dans la dernière epoch lancée)")
print(f"raison de l'arrêt     : {result['raison_arret']} (patience demandée : {config['patience']})")
if result["raison_arret"] == "arrêt anticipé":
    print("=> la loss val n'a pas battu son record pendant les dernières mesures : on s'arrête, best/ garde le record.")
else:
    print("=> pas d'arrêt anticipé : toutes les epochs ont été faites ; best/ contient quand même le record, pas le dernier état.")

### 5.4 Comparaison avec les essais précédents

Le tableau rassemble les entraînements v0.2 et v0.3 (dossiers `experiments/results/`, versionnés dans le dépôt) et le nouveau run. Entre v0.2 et v0.3, **un seul réglage a changé : le learning rate** (la carte graphique aussi, ce qui change la durée mais pas les valeurs). v1.0 reprend les réglages de v0.3 et affine l'arrêt : mesure de la loss val tous les 25 pas et choix du meilleur adaptateur au pas près.

Le tableau est généré par `write_runs_table` à partir des fichiers de chaque dossier. v0.2 n'ayant pas de `config.json`, son GPU et son learning rate sont donnés à la main.

In [ ]:
from IPython.display import Markdown
from src.training.plot_losses import write_runs_table

runs = [
    {"version": "v0.2", "output_dir": "experiments/results/train_v0.2", "gpu": "RTX 4070", "learning_rate": 2e-4},
    {"version": "v0.3", "output_dir": "experiments/results/train_v0.3"},
]
# le tableau lit historique_loss.csv : sans epoch complète, le run n'a pas de ligne à afficher
if len(pd.read_csv(f"{RUN_DIR}/historique_loss.csv")) > 0:
    runs.append({"version": RUN_NAME, "output_dir": RUN_DIR})
else:
    print("aucune epoch complète pour", RUN_NAME, ": run absent du tableau")
write_runs_table(runs, RUNS_TABLE)
Markdown(open(RUNS_TABLE, encoding="utf-8").read())

Remarques de lecture : la « loss val par epoch » est mesurée à la **fin** de chaque epoch complète. Pour v1.0, une epoch interrompue par l'arrêt anticipé n'a pas de colonne, et la meilleure mesure peut se situer en cours d'epoch (toutes les 25 pas) : elle est donnée par le §5.3 et par `resultat.json`. Les durées par epoch de v1.0 incluent les mesures de validation intermédiaires.

### 5.5 Constat sur v0.2 et v0.3

La cellule suivante lit les historiques des deux essais précédents et formule le constat à partir des données : loss val minimale de chaque run, et comportement de la loss val à la dernière epoch.

In [ ]:
def describe_run(version, folder):
    """Affiche un constat sur un run (loss val minimale, dernière epoch) ; retourne la loss val minimale."""
    history = pd.read_csv(f"{folder}/historique_loss.csv")
    best = history.loc[history["loss_val"].idxmin()]
    last = history.iloc[-1]
    trend = "remonte" if last["loss_val"] > best["loss_val"] else "ne remonte pas"
    print(f"{version} : loss val minimale {best['loss_val']:.4f} (epoch {int(best['epoch'])}) ; "
          f"à la dernière epoch ({int(last['epoch'])}) elle vaut {last['loss_val']:.4f} : elle {trend} par rapport au minimum.")
    return best["loss_val"]


minimums = {run["version"]: describe_run(run["version"], run["output_dir"]) for run in runs[:2]}
lower = min(minimums, key=minimums.get)
print(f"écart entre les deux minimums : {abs(minimums['v0.2'] - minimums['v0.3']):.4f} (le plus bas : {lower}).")
print("un seul entraînement par réglage : un petit écart ne permet pas de conclure sur le learning rate.")

## 6. Évaluation sur le jeu de validation (chiffres légèrement optimistes)

### 6.1 Ce qu'on mesure

La loss dit si le modèle prédit bien les mots de la réponse, pas si le **verdict** est juste. Un modèle peut avoir une bonne loss et se tromper de verdict. On fait donc répondre le modèle à chacun des exemples de `val`, sans lui montrer la réponse attendue, et on compare son verdict à celui attendu :

- **exactitude** : verdicts justes / nombre d'exemples ;
- **format valide** : part des réponses qui respectent le contrat de sortie (une première ligne `[VERDICT: ...]` ou `[HORS_PERIMETRE]`, puis un feedback) ;
- **matrice de confusion** : pour chaque verdict attendu, ce que le modèle a répondu.

La génération est **gloutonne** (toujours le token le plus probable) : la mesure est reproductible, et le modèle de base et le modèle adapté reçoivent exactement les mêmes prompts.

### 6.2 Lancer les deux évaluations

Le script charge le modèle, génère une réponse par exemple de `val`, puis écrit un fichier JSON sur Drive. Les lignes de progression sont masquées ; elles sont toutes dans le JSON (champ `rows`). Compter de quelques minutes à un quart d'heure par modèle selon le GPU.

In [ ]:
!python -u -m src.training.evaluate_verdicts --adapter none --split val --output {BASE_JSON} 2>&1 | grep --line-buffered -v "^exemple "

In [ ]:
!python -u -m src.training.evaluate_verdicts --adapter {BEST_DIR} --split val --output {LORA_JSON} 2>&1 | grep --line-buffered -v "^exemple "

### 6.3 Tableau synthétique : base contre LoRA

On lit les deux JSON et on calcule, pour chaque modèle : l'exactitude globale, le format valide, l'exactitude pour chaque verdict attendu, et le nombre d'**erreurs graves**. Une erreur grave est un cas où l'élève avait tort ou hors sujet (`INCORRECT` ou `HORS_PERIMETRE` attendu) et où le tuteur a répondu `PARFAIT` : il félicite une mauvaise réponse.

In [ ]:
def load_result(path):
    """Lit un JSON de evaluate_verdicts ; retourne le dict complet et ses lignes en DataFrame."""
    with open(path, encoding="utf-8") as file:
        data = json.load(file)
    return data, pd.DataFrame(data["rows"])


def accuracy_by_verdict(rows):
    """Exactitude (en %) pour chaque verdict attendu de VERDICTS ; 0 si ce verdict est absent."""
    values = []
    for verdict in VERDICTS:
        subset = rows[rows["expected"] == verdict]
        values.append(100 * subset["correct"].mean() if len(subset) else 0.0)
    return values


base_data, base_rows = load_result(BASE_JSON)
lora_data, lora_rows = load_result(LORA_JSON)
base_by_verdict = accuracy_by_verdict(base_rows)
lora_by_verdict = accuracy_by_verdict(lora_rows)

In [ ]:
def count_grave_errors(rows):
    """Nombre d'exemples INCORRECT ou HORS_PERIMETRE attendus que le tuteur a jugés PARFAIT."""
    return int((rows["expected"].isin(["INCORRECT", "HORS_PERIMETRE"]) & (rows["predicted"] == "PARFAIT")).sum())


summary = {"Exactitude globale": [], "Format valide": []}
for data in (base_data, lora_data):
    summary["Exactitude globale"].append(f"{data['number_of_correct']}/{data['number_of_examples']} ({100 * data['accuracy']:.1f} %)")
    summary["Format valide"].append(f"{100 * data['valid_format_rate']:.1f} %")
for verdict, base_value, lora_value in zip(VERDICTS, base_by_verdict, lora_by_verdict):
    size = int((base_rows["expected"] == verdict).sum())
    summary[f"Exactitude, attendu = {verdict} (n = {size})"] = [f"{base_value:.1f} %", f"{lora_value:.1f} %"]
summary["Erreurs graves (INCORRECT / HORS_PERIMETRE jugés PARFAIT)"] = [count_grave_errors(base_rows), count_grave_errors(lora_rows)]

pd.DataFrame(summary, index=["Base", "LoRA"]).T

### 6.4 Matrices de confusion

Chaque ligne est un verdict **attendu**, chaque colonne un verdict **prédit** ; la diagonale regroupe les réussites. `INVALIDE` désigne une sortie qui ne respecte pas le format.

In [ ]:
columns = VERDICTS + ["INVALIDE"]
matrices = {}
for name, data in (("Base", base_data), ("LoRA", lora_data)):
    matrices[name] = pd.DataFrame(data["confusion"]).T.reindex(index=VERDICTS, columns=columns).fillna(0).astype(int)

# côte à côte : une colonne principale par modèle
pd.concat(matrices, axis=1)

### 6.5 Exactitude par verdict

Même information que dans le tableau, en barres : orange pour le modèle de base, bleu pour le modèle adapté.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BACKGROUND, TEXT, TEXT_SECONDARY = "#fcfcfb", "#0b0b0b", "#52514e"

figure, axis = plt.subplots(figsize=(9, 4.5))
figure.patch.set_facecolor(BACKGROUND)
axis.set_facecolor(BACKGROUND)
positions = np.arange(len(VERDICTS))
width = 0.38
for shift, values, color, label in ((-width / 2, base_by_verdict, "#eb6834", "Base"), (width / 2, lora_by_verdict, "#2a78d6", "LoRA")):
    bars = axis.bar(positions + shift, values, width, color=color, label=label)
    axis.bar_label(bars, fmt="%.0f %%", color=TEXT_SECONDARY, fontsize=9, padding=2)
axis.set_xticks(positions, VERDICTS)
axis.set_ylim(0, 112)
axis.set_ylabel("exactitude (%)", color=TEXT_SECONDARY)
axis.set_title("Verdicts justes par verdict attendu, base contre LoRA (val)", color=TEXT, fontsize=12)
axis.tick_params(colors=TEXT_SECONDARY)
for side in ("top", "right"):
    axis.spines[side].set_visible(False)
axis.legend(frameon=False, labelcolor=TEXT_SECONDARY)
figure.savefig(BARS_PATH, dpi=200, facecolor=BACKGROUND, bbox_inches="tight")
plt.close(figure)
Image(BARS_PATH)

### 6.6 À lire avec précaution

- **Le chiffre de `val` est légèrement optimiste pour le modèle adapté.** Ces exemples ont servi à choisir le meilleur pas (celui de loss val minimale) : le modèle ne s'est pas entraîné dessus, mais le choix de l'adaptateur en dépend.
- **`val` est étroit et corrélé** (§2.2) : peu de questions, de nombreuses variantes. Un écart de quelques exemples change vite les pourcentages.
- **L'évaluation indépendante exigée par le sujet** (au moins 20 requêtes jamais vues, ni à l'entraînement ni pour choisir l'adaptateur) est celle du **jeu de test du bloc D, à brancher quand il est fusionné**.
- **Le verdict n'est pas toute la qualité.** On ne mesure ici ni les informations inventées, ni le respect de l'historique, ni la qualité pédagogique du feedback : ces critères relèvent de la **grille humaine du bloc D**.

## 7. Analyse de cas

Des chiffres globaux cachent le comportement réel. On regarde donc des cas précis.

### 7.1 Choisir six exemples

Le choix est calculé à partir des deux JSON, pas écrit à la main : jusqu'à 2 cas où le LoRA a raison et la base tort, 2 cas où les deux se trompent, 1 refus `HORS_PERIMETRE` bien traité par le LoRA, 1 cas où le LoRA se trompe. À chaque catégorie, on varie le verdict attendu quand c'est possible. Si une catégorie est vide, elle est simplement absente.

In [ ]:
merged = base_rows.merge(lora_rows, on=["index", "expected"], suffixes=("_base", "_lora"))
chosen = []


def pick(subset, number, category):
    """Ajoute à `chosen` jusqu'à `number` exemples de `subset` pas encore choisis, en variant le verdict attendu."""
    free = subset[~subset["index"].isin([item["index"] for item in chosen])]
    # un exemple par verdict attendu d'abord, puis le reste : on évite six cas du même verdict
    ordered = pd.concat([free.drop_duplicates("expected"), free]).drop_duplicates("index")
    for _, row in ordered.head(number).iterrows():
        chosen.append({"categorie": category, "index": int(row["index"]), "expected": row["expected"]})


pick(merged[merged["correct_lora"] & ~merged["correct_base"]], 2, "LoRA juste, base fausse")
pick(merged[~merged["correct_lora"] & ~merged["correct_base"]], 2, "les deux se trompent")
pick(merged[(merged["expected"] == "HORS_PERIMETRE") & merged["correct_lora"]], 1, "refus hors périmètre")
pick(merged[~merged["correct_lora"]], 1, "le LoRA se trompe")
pick(merged[merged["correct_lora"] & ~merged["correct_base"]], max(0, 6 - len(chosen)), "LoRA juste, base fausse")  # complète si une catégorie est vide

print(f"{len(chosen)} cas choisis")

### 7.2 Figer les cas

À la première exécution, la liste est écrite dans `cas_figes_<RUN_NAME>.json` sur Drive. Aux exécutions suivantes, on relit ce fichier : les mêmes cas sont analysés, et les commentaires rédigés au §7.5 restent valables.

In [ ]:
if os.path.exists(FROZEN_PATH):
    with open(FROZEN_PATH, encoding="utf-8") as file:
        chosen = json.load(file)
    print("cas relus depuis", FROZEN_PATH)
else:
    with open(FROZEN_PATH, "w", encoding="utf-8") as file:
        json.dump(chosen, file, ensure_ascii=False, indent=2)
    print("cas figés dans", FROZEN_PATH)
pd.DataFrame(chosen)

### 7.3 Générer les réponses de la base et du LoRA

On charge le modèle avec l'adaptateur `best/`. Pour obtenir la réponse du **modèle de base**, on désactive temporairement l'adaptateur avec `model.disable_adapter()` : le modèle 4 bits est alors identique au modèle de base, sans le recharger. Les deux générations sont gloutonnes, comme dans l'évaluation du §6.

In [ ]:
from src.training.model import load_model_with_adapter
from src.training.train import generate_answer

model, tokenizer = load_model_with_adapter(BEST_DIR)

cases = []
for item in chosen:
    conversation = val[item["index"]]
    prompt_messages = conversation[:-1]  # jamais la réponse attendue : le modèle ne doit pas la voir
    lora_text = generate_answer(model, tokenizer, prompt_messages)
    with model.disable_adapter():  # adaptateur coupé = modèle de base
        base_text = generate_answer(model, tokenizer, prompt_messages)
    cases.append({**item, "student": conversation[-2]["content"], "expected_text": conversation[-1]["content"],
                  "base_text": base_text, "lora_text": lora_text})

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print(len(cases), "cas générés")

### 7.4 Les cas côte à côte

Pour chaque cas : la réponse attendue, celle de la base et celle du LoRA. L'étiquette **réussite** ou **erreur** compare le verdict généré au verdict attendu. La ligne « cohérence » vérifie que la génération refaite ici donne le même verdict que celui enregistré dans le JSON du §6.

In [ ]:
import html
from IPython.display import HTML


def render_answer(title, text, expected, judged=True):
    """Colonne HTML d'une réponse, avec son verdict et (si judged) l'étiquette réussite / erreur."""
    verdict = label_of(text)
    ok = verdict == expected
    tag = f" : {'réussite' if ok else 'erreur'}" if judged else ""
    color = ("#1a7f37" if ok else "#c62828") if judged else "inherit"
    head = f"<b>{title}</b><br><b style='color:{color}'>{verdict}{tag}</b>"
    return f"<td style='vertical-align:top;width:33%;padding:6px'>{head}<br><span style='white-space:pre-wrap'>{html.escape(text)}</span></td>"


for number, case in enumerate(cases, start=1):
    row = merged[merged["index"] == case["index"]].iloc[0]
    same = label_of(case["base_text"]) == row["predicted_base"] and label_of(case["lora_text"]) == row["predicted_lora"]
    display(Markdown(f"#### Cas {number} : {case['categorie']} (val n°{case['index']}, attendu : {case['expected']})"))
    student = html.escape(case["student"].strip())
    display(HTML(f"<p style='white-space:pre-wrap'><b>Réponse de l'élève :</b> {student}<br><i>cohérence avec le JSON du §6 : {'oui' if same else 'NON (génération différente)'}</i></p>"))
    cells = [render_answer("Attendu", case["expected_text"], case["expected"], judged=False),
             render_answer("Base", case["base_text"], case["expected"]),
             render_answer("LoRA", case["lora_text"], case["expected"])]
    display(HTML("<table style='width:100%;border-collapse:collapse'><tr>" + "".join(cells) + "</tr></table>"))

### 7.5 Consignes d'analyse

> ## À COMPLÉTER AVANT LE RENDU : au moins 5 commentaires (réussites ou erreurs)
> Les commentaires ci-dessous sont à rédiger à la main après lecture des cas, puis le notebook est enregistré.

Pour chaque cas, on regarde :

1. **Le verdict** : est-il celui attendu ? Sinon, trop indulgent ou trop sévère ? Un `PARFAIT` donné à une mauvaise réponse est plus grave qu'un `PARTIEL` donné à la place d'un `INCORRECT`.
2. **Le format** : première ligne conforme au contrat, puis feedback sans balise supplémentaire.
3. **Le feedback donne-t-il la réponse ?** Un bon tuteur fait réfléchir sans donner la solution.
4. **Le ton** : bienveillant, adapté à l'élève, ni sec ni scolaire.
5. **Le refus** : sur une demande hors sujet, le modèle refuse brièvement et ramène vers la question.

Plusieurs feedbacks différents peuvent être bons : ce qui compte, c'est que le verdict soit juste et que l'élève avance.

**Commentaires (à rédiger) :**

- Cas 1 : à compléter après lecture.
- Cas 2 : à compléter après lecture.
- Cas 3 : à compléter après lecture.
- Cas 4 : à compléter après lecture.
- Cas 5 : à compléter après lecture.

## 8. Sauvegarde et rechargement

### 8.1 Recharger l'adaptateur depuis le disque

Une sauvegarde n'est utile que si on peut la relire. On recharge `best/` sur un modèle 4 bits neuf, on génère une réponse, et on vérifie qu'elle respecte le contrat de sortie avec `parse_pedagogical_response` (le même parseur que la baseline).

In [ ]:
from src.agents.baseline import parse_pedagogical_response

model, tokenizer = load_model_with_adapter(BEST_DIR)
answer = generate_answer(model, tokenizer, val[0][:-1])
parsed = parse_pedagogical_response(answer)

print(answer)
print("\nformat valide :", parsed["valid_format"], "| verdict :", parsed["verdict"])
print("OK : l'adaptateur rechargé produit une sortie conforme." if parsed["valid_format"] else "ATTENTION : sortie non conforme.")

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

### 8.2 Fichiers livrables et archive

On liste les fichiers produits avec leur taille, puis on crée une archive `.zip` du dossier `best/`. L'archive est pratique pour envoyer l'adaptateur à un autre bloc ou le télécharger depuis Drive.

In [ ]:
import shutil

deliverables = [BEST_DIR, f"{RUN_DIR}/config.json", f"{RUN_DIR}/resultat.json", f"{RUN_DIR}/historique_pas.csv",
                f"{RUN_DIR}/historique_val.csv", f"{RUN_DIR}/historique_loss.csv", PLOT_PATH, BARS_PATH,
                BASE_JSON, LORA_JSON, RUNS_TABLE]


def size_of(path):
    """Taille en Mo d'un fichier ou de tout un dossier."""
    if os.path.isfile(path):
        return os.path.getsize(path) / 1024**2
    return sum(os.path.getsize(os.path.join(folder, name)) for folder, _, names in os.walk(path) for name in names) / 1024**2


for path in deliverables:
    print(f"{size_of(path):8.2f} Mo  {path}" if os.path.exists(path) else f" MANQUANT    {path}")

print("\ncontenu de best/ :", sorted(os.listdir(BEST_DIR)))
zip_path = shutil.make_archive(ZIP_BASE, "zip", BEST_DIR)
print(f"archive : {zip_path} ({size_of(zip_path):.2f} Mo)")

### 8.3 Réutiliser l'adaptateur

L'adaptateur est un dossier (`best/`) qui ne contient que les petites matrices `A` et `B` ; le modèle de base est rechargé à part. Pour l'utiliser dans un autre notebook ou un autre bloc :

```python
from src.training.model import load_model_with_adapter
from src.training.train import generate_answer

model, tokenizer = load_model_with_adapter("chemin/vers/best")
answer = generate_answer(model, tokenizer, messages)  # messages = system + user, sans réponse du tuteur
```

Le moteur d'inférence commun `generate_response(..., model_version="lora")` est prévu pour charger ce dossier ; aujourd'hui il ne gère que la baseline (il lève `NotImplementedError` pour `"lora"`). Le brancher est un travail d'intégration à faire avec le bloc concerné, hors de ce notebook. Le dossier `best/` contient aussi `meilleur_pas.txt`, qui dit d'où vient cet adaptateur.

## 9. Limites et pistes d'amélioration

- **Labels générés par LLM** : les verdicts attendus ne sont pas une vérité humaine. Un audit humain plus large (accord entre annotateurs) renforcerait la confiance dans le dataset.
- **Validation étroite** : `val` ne contient que quelques questions ; le meilleur pas est choisi sur elles. Plus de questions dans `val` (ou une validation croisée par question) donnerait un choix de pas plus fiable.
- **Un seul entraînement par réglage** : on ne connaît pas la variabilité due à la graine. Il faudrait répéter chaque réglage plusieurs fois avant de conclure qu'un réglage est meilleur qu'un autre.
- **Learning rate et rang non optimisés** : seul le learning rate a été comparé (v0.2 contre v0.3), sur une seule valeur chacun. Le rang `r`, `alpha` et les couches ciblées sont des valeurs courantes, jamais mesurées ici. Un petit balayage serait à faire.
- **Évaluation du verdict seulement** : on ne mesure pas les informations inventées, l'oubli de l'historique ni la qualité pédagogique du feedback (grille humaine du bloc D).
- **Modèle de 3 milliards de paramètres** : compromis imposé par le GPU disponible. Un modèle plus grand pourrait mieux juger les cas limites `PARTIEL` / `INCORRECT`.
- **Pas de récupération de fiches (RAG)** : la fiche est collée dans le prompt. Avec beaucoup de concepts, il faudrait aller chercher la bonne fiche automatiquement.
- **Données plus variées** : de vraies réponses d'élèves, d'autres matières et d'autres styles de formulation réduiraient l'écart entre les données synthétiques et l'usage réel.

## 10. Annexe : reproductibilité

### 10.1 Commandes équivalentes en ligne de commande

Depuis la racine du dépôt, sur une machine avec un GPU Nvidia (PyTorch installé avant, voir l'en-tête de `requirements-qlora.txt`) :

```bash
pip install -r requirements-qlora.txt
python -m src.training.run_training --epochs 3 --learning-rate 1e-4 --patience 4 --eval-every 25 --output-dir adapters/v1.0
python -m src.training.evaluate_verdicts --adapter none --split val --output eval_base.json
python -m src.training.evaluate_verdicts --adapter adapters/v1.0/best --split val --output eval_lora.json
```

Les versions de `requirements-qlora.txt` ont été figées sur un poste local (Python 3.13). Sur Colab (Python 3.12), les versions affichées au §1.5 font foi.

### 10.2 Structure du dossier de sortie

| Fichier ou dossier | Contenu |
|---|---|
| `config.json` | réglages de l'entraînement, écrits avant de commencer |
| `resultat.json` | meilleur pas, meilleure epoch, loss val minimale, raison de l'arrêt, pas total, `epoch_partielle` et `pas_dans_epoch` (arrêt en cours d'epoch), durée |
| `historique_pas.csv` | loss train de chaque pas (`pas`, `epoch`, `loss_train`) |
| `historique_val.csv` | chaque mesure de loss val (`pas`, `epoch`, `loss_val`) |
| `historique_loss.csv` | une ligne par epoch **complète** (loss train moyenne, loss val de fin d'epoch, durée) |
| `epoch_N/` | adaptateur à la fin de l'epoch N (epochs complètes seulement) |
| `best/` | adaptateur de la meilleure loss val, avec `meilleur_pas.txt` |

Les évaluations (`eval_*.json`), les graphiques (`.png`), le tableau `comparaison_runs.md`, les cas figés du §7.2 et l'archive `.zip` sont dans `EVAL_DIR`.

### 10.3 Récupérer le code et les fichiers

Récupérer le code sur la bonne branche :

```bash
git clone --branch bloc-c-entrainement https://github.com/EmineOA/NLP-Transformers.git
cd NLP-Transformers
git pull   # pour mettre à jour un clone existant
```

Où trouver chaque produit : l'adaptateur (`best/`), la configuration, le résultat et les trois historiques CSV sont dans `SAVE_DIR/<RUN_NAME>` ; les évaluations JSON, les deux graphiques PNG, `comparaison_runs.md`, `cas_figes_<RUN_NAME>.json` et l'archive `.zip` de `best/` sont dans `EVAL_DIR`. Les essais précédents (v0.2, v0.3) sont versionnés dans `experiments/results/` du dépôt. Le §8.2 liste les fichiers réellement présents, avec leur taille.

### 10.4 Commande reconstituée depuis `config.json`

Cette cellule reconstruit la commande d'entraînement à partir des réglages réellement enregistrés : si elle diffère de celle du §4, c'est que le run n'a pas été lancé avec les mêmes arguments.

In [ ]:
print(
    "python -m src.training.run_training"
    f" --epochs {config['epochs']} --learning-rate {config['learning_rate']}"
    f" --patience {config['patience']} --eval-every {config['eval_every']}"
    f" --lora-rank {config['lora_rank']} --lora-alpha {config['lora_alpha']} --lora-dropout {config['lora_dropout']}"
    f" --output-dir {RUN_DIR}"
)
print("GPU utilisé :", config["gpu"])